# Agent Traces to SFT

This customer-configured notebook requires an explicitly prepared, source-traceable, conversation-disjoint dataset before it can submit or resume one GlobalStandard supervised training job. The historical row split is provenance only, not an accepted validation contract. Prepared artifacts have not been generated by the source-only fix.

The workflow reports service training metrics only. It performs no deployment, fine-tuned inference, base-model comparison, fresh trace generation or automatic data repair.

> Actual results may vary by model version, data, configuration, region availability, and service conditions.

## 1. Configure and import

Follow the central environment steps in managed_training/GETTING_STARTED.md and the offline preparation instructions in this module's README first. Copy this module's .env.template to its ignored .env. Configure your own project endpoint and FOUNDRY_MODEL only when proceeding to hosted operations; local data preparation/checks need no credentials.

The source model example is gpt-4.1-nano-2025-04-14; select a fine-tuning model available in your project. Dated identifiers pin dated versions; an optional FOUNDRY_MODEL_VERSION checks returned metadata when the service supplies it, but does not pin an undated alias.

Run from the module or notebooks directory. Blank polling settings use documented defaults. The notebook reads explicit outputs/prepared artifacts, never silently falls back to the original row-level split, and does not regenerate traces.

In [ ]:
MODULE_NAME = '04-agent-traces-to-sft'
HYPERPARAMETERS = {'n_epochs': 3, 'learning_rate_multiplier': 1.0, 'batch_size': 1}
TRAIN_FILE = 'train.jsonl'
VALIDATION_FILE = 'val.jsonl'
TEST_FILE = 'test.jsonl'
JOB_SUFFIX = '04-agent-traces-to-sft'

import csv
import hashlib
import json
import math
import os
import re
import sys
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if ROOT.name != MODULE_NAME or not (ROOT / "data").is_dir():
    raise RuntimeError(f"Run from {MODULE_NAME} or its notebooks directory.")
TRACK_ROOT = ROOT.parent
COOKBOOK_ROOT = TRACK_ROOT.parent
sys.path.insert(0, str(COOKBOOK_ROOT))
sys.path.insert(0, str(TRACK_ROOT))
sys.path.insert(0, str(ROOT))

from shared.auth import create_project_context
from shared.config import load_foundry_config
from stage05_runtime import monitor_fine_tuning_job, wait_for_file_processed
from trace_integrity import validate_prepared

if (ROOT / ".env").is_file():
    load_dotenv(ROOT / ".env")

def setting(name, default=""):
    value = os.environ.get(name, "").strip()
    return value if value else default

def positive_seconds(name, default):
    value = float(setting(name, str(default)))
    if not math.isfinite(value) or value <= 0:
        raise ValueError(f"{name} must be a finite positive number.")
    return value


MODEL = setting("FOUNDRY_MODEL")
EXPECTED_MODEL_VERSION = setting("FOUNDRY_MODEL_VERSION")
TRAINING_TYPE = setting("FOUNDRY_TRAINING_TYPE", "GlobalStandard")
if TRAINING_TYPE != "GlobalStandard":
    raise ValueError("This recipe requires FOUNDRY_TRAINING_TYPE=GlobalStandard.")
RUN_ID = setting("FOUNDRY_RUN_ID", "default")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", RUN_ID):
    raise ValueError("FOUNDRY_RUN_ID must be 1-64 letters, digits, underscores or hyphens.")
FILE_POLL_SECONDS = positive_seconds("FOUNDRY_FILE_POLL_SECONDS", 5)
JOB_POLL_SECONDS = positive_seconds("FOUNDRY_JOB_POLL_SECONDS", 60)
FILE_TIMEOUT_SECONDS = positive_seconds("FOUNDRY_FILE_TIMEOUT_SECONDS", 900)
JOB_TIMEOUT_SECONDS = positive_seconds("FOUNDRY_JOB_TIMEOUT_SECONDS", 86400)
RUN_DIR = ROOT / "outputs" / RUN_ID
STATE_PATH = RUN_DIR / "run-state.json"
# Prepared artifacts are explicit outputs, never an automatic repair of data/.
DATA = ROOT / "outputs" / "prepared"
print({"model": MODEL or "not configured", "training_type": TRAINING_TYPE, "recipe": HYPERPARAMETERS})

## 2. Verify source-derived preparation

The original 100-trace source and its historical 80/10/10 row split remain unchanged under data/ for provenance. The old split is not accepted as evaluation-isolated: generated call IDs can conceal identical conversations, and partial conversation snapshots can leak across row-level partitions.

Before running this notebook, explicitly run the offline preparation documented in the module README. It keeps every original trace, changes only the reviewed system/tool policy descriptions so inventory checks apply to exchanges and replacements, and groups shared user conversations and normalized message prefixes into a single partition. Exact prepared train/validation/test counts and hashes are derived by that command; 80/10/10 are targets, not promised output counts.

The cell below rebuilds the deterministic expected preparation from hash-pinned originals and checks prepared bytes, complete lineage and disjoint conversation/prefix identity without writing anything. Missing preparation fails closed. There are no agent calls or new generated answers; test data are never uploaded. This preparation and the offline tests have not been executed as part of the source-only change.

In [ ]:
# Read-only: independently rebuild expected preparation from hash-pinned sources.
# Missing preparation, fixture drift or conversation/prefix leakage raises here.
datasets, lineage = validate_prepared(DATA)
EXPECTED = {name: {"rows": details["rows"], "sha256": details["sha256"]}
            for name, details in lineage.items()}
VALIDATED_PREPARATION_SHA256 = hashlib.sha256(
    (DATA / "preparation-manifest.json").read_bytes()).hexdigest()
print(json.dumps({"prepared_data": lineage,
                  "conversation_and_prefix_isolation": "validated locally"}, indent=2))

## 3. Submission state and result parsing

Runtime evidence stays under ignored `outputs/<run-id>`. A blank `FOUNDRY_RUN_ID` uses `default`, so a rerun resumes the same job. Set a new run ID only for an intentional new paid run. A stored signature prevents reuse across project/model/data changes. Uploaded IDs are persisted before processing waits, and every resumed upload is checked again. An interrupted submission without a saved job ID requires explicit reconciliation with the service rather than a blind duplicate submission.

Each service metric column is kept separate; sampled and full validation metrics are not spliced into one misleading curve. Invalid/non-finite values and missing loss observations are explicit errors.


In [ ]:
from stage05_runtime import (
    create_job_once, load_run_state, submission_lock, upload_file_once, write_json_atomic,
)

def write_json(path, value):
    return write_json_atomic(path, value, outputs_root=ROOT / "outputs")

def load_state(signature):
    return load_run_state(STATE_PATH, signature, file_names=(TRAIN_FILE, VALIDATION_FILE))

def upload_once(client, state, name):
    return upload_file_once(
        client, state, name, data_dir=DATA, state_path=STATE_PATH,
        outputs_root=ROOT / "outputs", expected_sha256=lineage[name]["sha256"],
        poll_interval_seconds=FILE_POLL_SECONDS, timeout_seconds=FILE_TIMEOUT_SECONDS)

def submit_or_resume_job(client, state):
    # Caller holds submission_lock from before load_state through the saved job ID.
    if state.get("job_id"):
        print(f"Resuming {state['job_id']}; no new submission.")
        return state["job_id"]
    training = upload_once(client, state, TRAIN_FILE)
    validation = upload_once(client, state, VALIDATION_FILE)
    job_id = create_job_once(
        client, state, state_path=STATE_PATH, outputs_root=ROOT / "outputs",
        model=MODEL, training_file=training.id, validation_file=validation.id,
        method={"type": "supervised",
                "supervised": {"hyperparameters": HYPERPARAMETERS}},
        suffix=JOB_SUFFIX, extra_body={"trainingType": TRAINING_TYPE})
    print(f"Submitted {job_id}.")
    return job_id

def summarize_metrics(paths):
    series = {}
    for path in paths:
        with path.open(encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream)
            columns = [column for column in (reader.fieldnames or [])
                       if "loss" in column.casefold() or "token_accuracy" in column.casefold()]
            for row_number, row in enumerate(reader, 2):
                for column in columns:
                    raw = row.get(column)
                    if raw is None or not raw.strip():
                        continue
                    value = float(raw)
                    step = float(row["step"])
                    if not math.isfinite(value) or not math.isfinite(step):
                        raise ValueError(f"{path.name}:{row_number}: non-finite {column} or step.")
                    series.setdefault(column, []).append({"step": step, "value": value})
    if not any("loss" in column.casefold() and points for column, points in series.items()):
        raise RuntimeError("Service result CSVs contained no loss observations; no effectiveness conclusion can be drawn.")
    metrics = {}
    for column, points in series.items():
        values = [point["value"] for point in points]
        metrics[column] = {"observations": len(values), "first": values[0], "final": values[-1],
                           "minimum": min(values), "maximum": max(values),
                           "change": values[-1] - values[0] if len(values) > 1 else None,
                           "curve": points}
    return metrics

## 4. Opt in, upload, train, monitor and download

Executing this cell performs hosted calls and can incur training charges. The source hyperparameters above are retained (an empty dictionary keeps the source service-default recipe). The project client uses `DefaultAzureCredential`; uploads and fine-tuning use `AIProjectClient.get_openai_client()`. Failed/cancelled jobs and bounded wait timeouts raise clear errors. All clients close even on failure.

Creation POSTs disable SDK retries with `max_retries=0`, matching the shared duplicate-avoidance policy. A recorded pending job is monitored without another creation attempt; ambiguous submissions require reconciliation. The stored signature checks the requested project, model, data and recipe. Authentication and service errors propagate without fallback.


In [ ]:
# Recheck the source-derived preparation immediately before any hosted call.
# Running this cell without the read-only validation cell fails closed.
current_datasets, current_lineage = validate_prepared(DATA)
current_preparation_sha256 = hashlib.sha256(
    (DATA / "preparation-manifest.json").read_bytes()).hexdigest()
if (current_preparation_sha256 != VALIDATED_PREPARATION_SHA256
        or current_lineage != lineage or current_datasets != datasets):
    raise RuntimeError("Prepared data changed after validation; rerun the local checks.")
if not MODEL:
    raise ValueError("Set FOUNDRY_MODEL to a fine-tuning model identifier available in your project.")
print("This workflow uses paid cloud resources and may incur charges.")
config = load_foundry_config(environ=dict(os.environ))
signature = {
    "project_endpoint": config.project_endpoint, "model": MODEL,
    "expected_model_version": EXPECTED_MODEL_VERSION,
    "training_type": TRAINING_TYPE, "hyperparameters": HYPERPARAMETERS,
    "data": {name: details["sha256"] for name, details in EXPECTED.items()},
    "preparation_sha256": current_preparation_sha256,
}

# Lock before loading state, and retain it through every pending POST and ID write.
with submission_lock(STATE_PATH, outputs_root=ROOT / "outputs"):
    state = load_state(signature)
    # Waiting for another writer must not leave the preparation check stale.
    current_datasets, current_lineage = validate_prepared(DATA)
    current_preparation_sha256 = hashlib.sha256(
        (DATA / "preparation-manifest.json").read_bytes()).hexdigest()
    if (current_preparation_sha256 != VALIDATED_PREPARATION_SHA256
            or current_lineage != lineage or current_datasets != datasets):
        raise RuntimeError("Prepared data changed after validation; rerun the local checks.")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    write_json(RUN_DIR / "data-lineage.json", lineage)

    # All hosted resources close on failure; the lock also protects result writes.
    with create_project_context(config) as context:
        client = context.openai_client
        job_id = submit_or_resume_job(client, state)
        job = monitor_fine_tuning_job(
            client, job_id, poll_interval_seconds=JOB_POLL_SECONDS,
            timeout_seconds=JOB_TIMEOUT_SECONDS)
        resolved_version = (job.metadata or {}).get("model_version")
        if EXPECTED_MODEL_VERSION and resolved_version and str(resolved_version) != EXPECTED_MODEL_VERSION:
            raise RuntimeError(f"Service resolved model version {resolved_version}, expected {EXPECTED_MODEL_VERSION}.")
        write_json(RUN_DIR / "job.json", job.model_dump(mode="json"))
        if not job.result_files:
            raise RuntimeError("Succeeded job returned no result files; service loss metrics are unavailable.")
        result_paths = []
        for index, file_id in enumerate(job.result_files):
            response = client.files.content(file_id=file_id)
            content = response.read()
            if not isinstance(content, bytes):
                raise TypeError(f"Result file {file_id} did not return bytes.")
            path = RUN_DIR / f"result-{index}.csv"
            path.write_bytes(content)
            result_paths.append(path)
        metrics = summarize_metrics(result_paths)
        summary = {"status": str(job.status), "trained_tokens": job.trained_tokens,
                   "lineage": lineage, "service_training_metrics": metrics}
        write_json(RUN_DIR / "training-summary.json", summary)
print(json.dumps(summary, indent=2))

## 5. Training-metrics-only result

Plot the fresh service loss columns and read the downloaded CSVs and `training-summary.json`. Token accuracy is included only when emitted by the service. Missing metrics are not inferred, and prior local runs are not used as evidence.


In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots()
for column, details in metrics.items():
    if "loss" in column.casefold():
        points = details["curve"]
        axis.plot([point["step"] for point in points], [point["value"] for point in points],
                  marker="o", label=column)
axis.set(xlabel="Training step", ylabel="Service loss", title="Service training and validation loss")
axis.grid(alpha=0.25)
axis.legend()
figure.tight_layout()
figure.savefig(RUN_DIR / "service-loss.png", dpi=160)
plt.show()
plt.close(figure)
